# Simple: Aggregate Records with PAMOLA.CORE

**Goal**: Learn record aggregation basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply grouping and aggregation using execute()
- Compare before/after results
- Understand data reduction and utility preservation

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/aggregate_records/
        └── 01_aggregate_records_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd 
import numpy as np 
import sys 
import os 
import json 
from pathlib import Path 
from datetime import datetime 
from IPython.display import Image, display 
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.grouping.aggregate_records_op import AggregateRecordsOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data if file missing
- Review dataset structure before proceeding

**What you'll see:**
- Dataset summary (100 records, 7 columns)
- First 5 records with sales transaction data
- Column statistics: 5 categories, 5 regions, numeric amounts
- Fields suitable for aggregation: product_category, region, sale_amount, quantity

**Note:** Sample includes transaction-level data demonstrating grouping and aggregation use cases

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample sales transaction data
    np.random.seed(42)
    sample_data = pd.DataFrame({
        'transaction_id': range(1, 101),
        'product_category': np.random.choice(
            ['Electronics', 'Clothing', 'Food', 'Books', 'Home'], 
            size=100
        ),
        'region': np.random.choice(
            ['North', 'South', 'East', 'West', 'Central'], 
            size=100
        ),
        'sale_amount': np.random.uniform(10, 500, size=100).round(2),
        'quantity': np.random.randint(1, 20, size=100),
        'discount_applied': np.random.choice([True, False], size=100),
        'customer_satisfaction': np.random.randint(1, 6, size=100),
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]) or df[col].dtype == 'object':
        print(f"  {col:25s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_bool_dtype(df[col]):
        print(f"  {col:25s} ({dtype_str:10s}): {df[col].sum()} True, {(~df[col]).sum()} False")
    else:
        print(f"  {col:25s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE group_by_fields** in `create_config_kwargs()`
   - Default: `["location_city"]`
   - Change to YOUR dataset's grouping fields
2. **CUSTOMIZE aggregations** dictionary
   - Map fields to aggregation functions: sum, mean, count, min, max, std, median
3. Run to validate configuration and setup environment

**What you'll see:**
- Group by fields listed with unique group count
- Aggregations configured per field with functions
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and progress tracker ready (✓)

**Note:** Aggregation reduces record count while preserving statistical utility for analysis

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "group_by_fields": ["location_city"],
        "aggregations": {
            "current_salary_cad": ["sum", "mean", "count"],
        }
    }

kwargs = create_config_kwargs()
group_by_fields = kwargs["group_by_fields"]
aggregations = kwargs["aggregations"]

# Validate fields exist
print(f"\n🔍 Validating field configuration...\n")
missing_fields = [f for f in group_by_fields if f not in df.columns]
if missing_fields:
    raise ValueError(
        f"❌ Group by fields not found in dataset: {missing_fields}\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'group_by_fields' in create_config_kwargs()"
    )

missing_agg_fields = [f for f in aggregations.keys() if f not in df.columns]
if missing_agg_fields:
    raise ValueError(
        f"❌ Aggregation fields not found in dataset: {missing_agg_fields}\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'aggregations' in create_config_kwargs()"
    )

print(f"✓ Group by fields: {group_by_fields}")
print(f"  Number of unique groups: {df.groupby(group_by_fields).ngroups}")
print(f"\n✓ Aggregations configured:")
for field, funcs in aggregations.items():
    print(f"  {field}: {', '.join(funcs)}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="aggregation",
    description=f"Aggregation by {', '.join(group_by_fields)}",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description="Processing aggregation",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration parameters below
- Run to execute record aggregation
- Monitor progress bar (6 steps: load → group → aggregate → save → complete)

**Key parameters:**
- `group_by_fields`: Categorical fields for grouping
- `aggregations`: Map fields to functions (sum, mean, count, min, max, std, median)
- `custom_aggregations=None`: Optional custom functions

**What you'll see:**
- Configuration summary with group and aggregation counts
- Progress bar: load → validate → group by → apply aggregations → save → complete
- Completion status: "completed" (verify no errors)
- Artifact count (3-5 files expected)

**Note:** Aggregation reduces granularity while maintaining statistical properties for privacy and efficiency

In [ ]:
# Create operation with production-style configuration
operation = AggregateRecordsOperation(
    # Core parameters
    group_by_fields=group_by_fields,
    aggregations=aggregations,
    custom_aggregations=None,  # Optional: custom aggregation functions
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_dask=False,
    npartitions=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Group by:         {operation.group_by_fields}")
print(f"  Aggregations:     {len(operation.aggregations)} fields")
print(f"  Custom aggs:      {operation.custom_aggregations or 'None'}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing record aggregation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load aggregated data from task_dir
- Review reduction metrics and group statistics
- Compare original vs aggregated record counts

**What you'll see:**
- Sample aggregated records (first 10 groups)
- Reduction metrics: reduction ratio and percentage
- Summary: original records, aggregated groups, field counts
- Key metrics: execution time, records per second

**Note:** Lower record count improves privacy (k-anonymity) and processing efficiency

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample aggregated records
    print("\n🔍 Sample Aggregated Records (first 10):")
    print(result_df.head(10).to_string())
    
    # Calculate reduction metrics
    original_records = len(df)
    aggregated_records = len(result_df)
    reduction_ratio = aggregated_records / original_records
    reduction_pct = (1 - reduction_ratio) * 100
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:     {original_records}")
    print(f"  Aggregated groups:    {aggregated_records}")
    print(f"  Reduction ratio:      {reduction_ratio:.4f}")
    print(f"  Records reduced by:   {reduction_pct:.1f}%")
    print(f"  Original fields:      {len(df.columns)}")
    print(f"  Result fields:        {len(result_df.columns)}")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Fewer records = Better data reduction while preserving utility!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-2 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Aggregated CSV (grouped records)
├── metrics/          # Aggregation and reduction metrics JSON
├── visualizations/   # Group distribution charts (PNG)
└── config/           # Operation configuration
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance and aggregation statistics
2. **Output Data**: Aggregated records with sample rows
3. **Visualizations**: Charts showing group distributions and aggregation results

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # -------- METADATA --------
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")

            op = metadata.get("operation", {})
            print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")

            # -------- AGGREGATION METRICS --------
            print("\n📊 AGGREGATION METRICS:")
            print("-" * 80)

            print(f"   Total Input Records:   {metrics.get('total_input_records', 'N/A')}")
            print(f"   Total Output Records:  {metrics.get('total_output_records', 'N/A')}")
            print(f"   Total Input Fields:    {metrics.get('total_input_fields', 'N/A')}")
            print(f"   Total Output Fields:   {metrics.get('total_output_fields', 'N/A')}")
            print(f"   Number of Groups:      {metrics.get('num_groups', 'N/A')}")

            reduction_ratio = metrics.get("reduction_ratio")
            print(
                f"   Reduction Ratio:       {reduction_ratio:.4f}"
                if isinstance(reduction_ratio, (int, float)) and pd.notna(reduction_ratio)
                else "   Reduction Ratio:       N/A"
            )

            # -------- GROUP SIZE STATISTICS --------
            print("\n📏 GROUP SIZE STATISTICS:")
            print("-" * 80)

            print(f"   Min Group Size:        {metrics.get('group_size_min', 'N/A')}")
            print(f"   Max Group Size:        {metrics.get('group_size_max', 'N/A')}")

            mean_size = metrics.get("group_size_mean")
            print(
                f"   Mean Group Size:       {mean_size:.2f}"
                if isinstance(mean_size, (int, float)) and pd.notna(mean_size)
                else "   Mean Group Size:       N/A"
            )

            median_size = metrics.get("group_size_median")
            print(
                f"   Median Group Size:     {median_size:.2f}"
                if isinstance(median_size, (int, float)) and pd.notna(median_size)
                else "   Median Group Size:     N/A"
            )

            # -------- AGGREGATED FIELD STATISTICS --------
            agg_stats = metrics.get("aggregated_field_stats", {})
            if agg_stats:
                print("\n📈 AGGREGATED FIELD STATISTICS:")
                print("-" * 80)

                for field, stats in agg_stats.items():
                    print(f"\n   {field}:")

                    for key, label in [
                        ("min", "Min"),
                        ("max", "Max"),
                        ("mean", "Mean"),
                        ("median", "Median"),
                        ("std", "Std Dev"),
                    ]:
                        val = stats.get(key)
                        if isinstance(val, (int, float)) and pd.notna(val):
                            print(f"      {label}: {val:.2f}")
                        else:
                            print(f"      {label}: N/A")

            # -------- PERFORMANCE --------
            print("\n⚡ PERFORMANCE:")
            print("-" * 80)

            exec_time = metrics.get("execution_time_seconds")
            print(
                f"   Execution Time:   {exec_time:.4f}s"
                if isinstance(exec_time, (int, float)) and pd.notna(exec_time)
                else "   Execution Time:   N/A"
            )

            rps = metrics.get("records_per_second")
            print(
                f"   Records/Second:   {rps:.2f}"
                if isinstance(rps, (int, float)) and pd.notna(rps)
                else "   Records/Second:   N/A"
            )
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (NEWEST)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            # Show aggregation summary by group
            print(f"\n\n📊 Aggregation Summary:")
            print("-" * 80)
            print(f"Total groups: {len(output_df)}")
            print(f"\nSample aggregated values for first 5 groups:")
            for idx, row in output_df.head(5).iterrows():
                group_keys = [f"{field}={row[field]}" for field in group_by_fields if field in row]
                print(f"\n  Group: {', '.join(group_keys)}")
                for col in output_df.columns:
                    if col not in group_by_fields:
                        print(f"    {col}: {row[col]}")
            
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure aggregation with group by fields and aggregation functions  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Group records by categorical fields for aggregation
- Apply multiple aggregation functions (sum, mean, count, etc.) per field
- Reduce dataset size while preserving statistical utility
- Standard aggregations: count, sum, mean, median, min, max, std, var, first, last, nunique
- All artifacts saved in structured directories

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_aggregate_records_advanced.ipynb`** includes:
- Custom aggregation functions
- Multi-level grouping strategies
- Handling missing values in aggregations
- Advanced statistical aggregations
- Performance optimization for large datasets
- Production deployment patterns

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)